# Project Aurelius: Electrolyte Design Tutorial

This notebook demonstrates the full Project Aurelius workflow for discovering
battery electrolyte candidates using a hybrid physics-based oracle with
adaptive evolutionary search.

## What You'll Learn

1. **Screen a single molecule** through the Filter → Oracle → Score pipeline
2. **Evaluate a binary mixture** with thermodynamic mixing rules and synergy bonus
3. **Run the discovery loop** with tournament or NSGA-II selection
4. **Inspect retrosynthetic depth** and synthetic accessibility
5. **Use the experimental feedback controller** for on-line oracle refinement

## Requirements

```bash
pip install -e '.[dev]'
```
This notebook assumes you are running from the ProjectAurelius root directory.

## 1. Single-Molecule Screening

In [ ]:
from aurelius.pipeline import AureliusPipeline
from aurelius.types import MoleculeContext

pipeline = AureliusPipeline()
pipeline.initialize()

smiles = 'CC1=CC=CC=C1'
ctx = MoleculeContext.from_smiles(smiles)
result = pipeline.screen_molecule(ctx)

print(f"SMILES: {result['tier2']['smiles'] if 'smiles' in result.get('tier2', {}) else smiles}")
print(f"HOMO: {result['tier2']['homo_eV']:.3f} eV")
print(f"LUMO: {result['tier2']['lumo_eV']:.3f} eV")
print(f"Dielectric proxy: {result['tier2']['dielectric_proxy']:.3f}")
print(f"Viscosity proxy: {result['tier2']['viscosity_proxy']:.3f}")
print(f"Li solvation proxy: {result['tier2']['li_solvation_proxy']:.3f}")
print(f"Conformal confidence: {result['tier2']['conformal_confidence']:.4f}")
print(f"SA score: {result['score']['sa_score']:.3f}")
print(f"Synthesis depth: {result['score']['synthesis_depth']}")
print(f"Total score: {result['score']['total_score']:.1f}/100")
print(f"Viable: {result['score']['is_viable']}")

## 2. Binary Mixture Screening

Mixtures are scored using ideal mixing rules plus a non-linear synergy
bonus when components are complementary (high-dielectric + low-viscosity).

In [ ]:
ctx1 = MoleculeContext.from_smiles('CC1=CC=CC=C1')
ctx2 = MoleculeContext.from_smiles('CC(=O)OC')
mix = pipeline.screen_mixture(ctx1, ctx2, frac1=0.5)

mp = mix['mixture_properties']
print(f"Mixture dielectric: {mp['dielectric_proxy']:.3f}")
print(f"Mixture viscosity: {mp['viscosity_proxy']:.3f}")
print(f"Synergy bonus: {mp['synergy_bonus']:.4f}")
print(f"Mixture total score: {mix['score']['total_score']:.1f}/100")

## 3. Full Discovery Loop

The `DiscoveryLoop` runs mutation → evaluation → selection → evolution
autonomously. Two selection strategies are available:

- **Tournament selection** (default): single-objective, Tanimoto diversity penalty
- **NSGA-II** (`--nsga2`): multi-objective Pareto optimization across 7 properties

In [ ]:
from aurelius.agent.loop import AgentConfig, run_screening

cfg = AgentConfig(max_generations=10, batch_size=15)
results = run_screening(cfg)

print(f"Total screened: {results['total_screened']}")
print(f"Viable discoveries: {results['total_viable']}")
print(f"Invalid discarded: {results['total_invalid']}")

### 3b. NSGA-II Multi-Objective Mode

Activates Pareto-optimal selection across dielectric (max), viscosity (min),
Li solvation (max), orbital energies, and synthetic accessibility.

In [ ]:
from aurelius.agent.loop import AgentConfig, run_screening

cfg_nsga2 = AgentConfig(max_generations=10, batch_size=15, use_nsga2=True)
results = run_screening(cfg_nsga2)

## 4. Retrosynthetic Depth & Synthetic Accessibility

Every molecule receives a synthesis depth score (1–5) indicating how
many BRICS retrosynthesis steps are needed to reach commercial precursors.
This is computed via `combined_grounding_score` and exposed in
`ScreeningResult.synthesis_depth`.

In [ ]:
from aurelius.agent.mutation.retrosynthetic import (
    brics_retrosynthetic_depth,
    get_commercial_precursor_count,
)
from aurelius.types import MoleculeContext

smiles_list = ['CCO', 'CC(=O)OC', 'CC1=CC=CC=C1', 'O=S(=O)(C)C']
for smi in smiles_list:
    ctx = MoleculeContext.from_smiles(smi)
    if ctx is None:
        continue
    depth = brics_retrosynthetic_depth(ctx.mol)
    print(f"{smi:20s} → depth={depth}")

print(f"\nCommercial precursors in database: {get_commercial_precursor_count()}")

## 5. Experimental Feedback Controller

The `FeedbackController` accumulates screening data and periodically
refits the Gaussian Process residual model (DeltaCorrection) to keep
the oracle calibrated to the explored chemical space.

You can also supply experimental ground-truth values to further refine
the oracle during an active learning campaign.

In [ ]:
from aurelius.agent.feedback import FeedbackController

fc = FeedbackController(refit_interval=3)
fc.accumulate(
    smiles='CCO',
    homo_prediction=-8.5,
    lumo_prediction=-1.2,
    homo_corrected=-7.8,
    luto_corrected=-0.9,
    total_score=72.5,
    conformal_confidence=0.96,
    generation=1,
    experimental_homo=-7.9,
    experimental_lumo=-1.0,
)

print(f"Records accumulated: {fc.num_records}")

info = fc.maybe_refit(current_generation=5)
print(f"Refit info: {info}")

## 6. Inspect Discoveries

After the loop finishes, discoveries are saved as an SDF file with
full property annotations.

In [ ]:

discoveries = sorted(results['discoveries'], key=lambda r: -r.total_score)
print(f"Total discoveries: {len(discoveries)}")

for d in discoveries[:5]:
    print(
        f"  {d.smiles:40s} score={d.total_score:5.1f} "
        f"dielectric={d.dielectric_proxy:.2f} "
        f"viscosity={d.viscosity_proxy:.2f} "
        f"depth={d.synthesis_depth}"
    )